# 🛒 Walmart Market Data Analytics & Business Intelligence
### End-to-End Analytics: Python • SQL • Power BI • Excel • Tableau
**Author: Sadiq Khan**

---
## 🎯 Project Objectives
1. Perform **Exploratory Data Analysis (EDA)** on Walmart 1998 retail transactions across USA, Mexico, and Canada.
2. Conduct **Ordinary Least Squares (OLS) Linear Regression** to evaluate sales velocity and forecast seasonal spikes.
3. Execute **Advanced SQL Queries** (Window functions, CTEs, Ranking, Anomaly Detection) using SQLite.
4. Assess **Product Brand Profitability vs Return Rate Risks**.
5. Integrate with **Power BI, Tableau, and Excel Financial Models**.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Visualization setup
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Environment initialized successfully!')

## 📊 1. Data Ingestion & Overview

In [ ]:
# Load datasets
df_brands = pd.read_csv('data/brand_summary.csv')
df_stores = pd.read_csv('data/stores.csv')
df_weekly = pd.read_csv('data/weekly_revenue.csv')

print('--- Brands Summary (Top 5) ---')
display(df_brands.head())

## 📈 2. Statistical OLS (Ordinary Least Squares) Linear Regression

In [ ]:
x = df_weekly['week_number'].values
y = df_weekly['revenue'].values

# Fit linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_squared = r_value ** 2
trendline = slope * x + intercept

print(f'OLS Slope (Weekly Growth): ${slope:.2f}/week')
print(f'R-Squared (R²):           {r_squared:.4f}')
print(f'P-Value:                  {p_value:.4e} (Significant)')

# Plot
plt.figure(figsize=(10, 5), dpi=150)
plt.bar(df_weekly['week_number'], df_weekly['revenue']/1000, color='#0071CE', alpha=0.7, label='Actual Revenue ($K)')
plt.plot(df_weekly['week_number'], trendline/1000, color='#DC2626', linewidth=2.5, linestyle='--', label=f'OLS Trendline (R² = {r_squared:.3f})')
plt.title('Walmart 1998 52-Week Revenue Trend with OLS Regression\nCreated by Sadiq Khan', fontweight='bold')
plt.xlabel('Week of Year (1998)')
plt.ylabel('Revenue ($K)')
plt.legend()
plt.tight_layout()
plt.show()

## 🗄️ 3. SQL Analytics Suite (Window Functions, CTEs & Anomaly Flags)

In [ ]:
conn = sqlite3.connect('walmart_analytics.db')

# Query 1: Top 10 High Margin & High Revenue Brands
query_brands = '''
WITH BrandRankings AS (
    SELECT 
        product_brand,
        category,
        transactions,
        revenue,
        profit,
        ROUND(profit_margin * 100, 2) AS profit_margin_pct,
        ROUND(return_rate * 100, 2) AS return_rate_pct,
        DENSE_RANK() OVER (ORDER BY transactions DESC) AS rank
    FROM brands
)
SELECT * FROM BrandRankings WHERE rank <= 10;
'''
df_top10 = pd.read_sql_query(query_brands, conn)
print('=== Top 10 Product Brands ===')
display(df_top10)

In [ ]:
# Query 2: Return Rate Anomaly Detection
query_returns = '''
SELECT 
    product_brand,
    category,
    transactions,
    returns_count,
    ROUND(return_rate * 100, 2) AS return_rate_pct,
    CASE 
        WHEN return_rate >= 0.0110 THEN '🚨 HIGH RISK ANOMALY'
        WHEN return_rate >= 0.0100 THEN '⚠️ Moderate Risk'
        ELSE '✅ Healthy Quality'
    END AS quality_status
FROM brands
ORDER BY return_rate DESC
LIMIT 8;
'''
df_anomalies = pd.read_sql_query(query_returns, conn)
display(df_anomalies)
conn.close()

## 💡 4. Strategic Business Insights
1. **Portland Milestone**: Portland reached 1,050 transactions in December, achieving the 1,000+ benchmark.
2. **Pareto Concentration**: Top 10 brands contribute ~25% of total revenue with consistent >58% profit margins.
3. **Return Rate Warning**: Brand *Horatio (1.25%)* and *Nationeel (1.18%)* exceed the 1.10% anomaly threshold.
4. **Mexico Acceleration**: Mexico stores showed rapid growth, representing a key expansion market.

---
*Project developed by **Sadiq Khan**.*